<a href="https://colab.research.google.com/github/LucasMartinscode/LucasMartinscode/blob/LucasMartinscode-patch-1/Aula_12.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

PG operator

In [ ]:
import requests
import pandas as pd
from io import BytesIO
from datetime import datetime

from airflow.models.baseoperator import BaseOperator
from custom_s3_hook import CustomS3Hook
from airflow.providers.postgres.hooks.postgres import PostgresHook

class BIPgOperator(BaseOperator):
    def __init__(self, url: str, tablename: str, **kwargs) -> None:
        super().__init__(**kwargs)
        self.url = url
        self.tablename = tablename
        self.custom_s3 = CustomS3Hook(bucket="covid-data")
        self.pg_hook = PostgresHook(postgres_conn_id="pg_awari")
        self.pg_conn = self.pg_hook.get_conn()
        self.engine = self.pg_hook.get_sqlalchemy_engine()
        self.current_time = datetime.now()
        self.current_date = self.current_time .strftime("%Y-%m-%d")

    def execute(self, context):
        self.process_to_pg()
        return self.url

    def process_to_pg(self):
        print("Fazendo download do arquivo: " + self.url)

        csv = self.custom_s3.get_object(key=f"datalake/{self.url}")
        df = pd.read_csv(csv, header=0)
        df.to_sql(self.tablename, con=self.engine, if_exists='replace', index=False)


DAGS

In [ ]:
import os
from airflow import DAG
from datetime import datetime
from bi_pg_operator import BIPgOperator

COVID_DATA_FILES = {
    'covid_cases_and_deaths': 'COVID-19 Cases and deaths - WHO.csv',
   'biweekly_cases': 'biweekly_cases.csv',
   'biweekly_cases_per_million': 'biweekly_cases_per_million.csv',
   'biweekly_deaths': 'biweekly_deaths.csv',
   'biweekly_deaths_per_million':'biweekly_deaths_per_million.csv',
   'full_data':'full_data.csv',
   'new_cases':'new_cases.csv',
   'new_cases_per_million':'new_cases_per_million.csv',
   'new_deaths':'new_deaths.csv'
   'new_deaths_per_million':'new_deaths_per_million.csv',
   'total_cases':'total_cases.csv',
   'total_cases_per_million':'total_cases_per_million.csv',
   'total_deaths':'total_deaths.csv',
   'total_deaths_per_million':'total_deaths_per_million.csv',

}

default_args = {
    'owner': 'airflow',
    'depends_on_past': False,
    'start_date': datetime(2021, 1, 1),
    'retries': 1,
}

for file_key, file_name in COVID_DATA_FILES.items():
    dag = DAG(
        dag_id=f"bi_{file_key}_dag",
        default_args=default_args,
        schedule_interval=None,
        catchup=False
    )

    download_task = BIPgOperator(
        task_id=f"download_{file_name}",
        url=file_name,
        tablename=file_key,
        dag=dag
    )

    globals()[f'bi_{file_key}_dag'] = dag
